In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DATA_DIR = Path("../../tables/parquet_2026-10-24/silver-layer-parquet")          # folder that holds the .parquet files
OUT_DIR = Path("../../tables/parquet_2026-10-24/gold-layer-parquet")       # cleaned outputs are written here
OUT_DIR.mkdir(exist_ok=True)
LOCAL_TZ = "Europe/Oslo"      # matches agency_timezone in gtfs_agency

# Which service dates of the real-time feed to analyse. None = every date found in the feed.
ET_DATES = None               # e.g. ["2026-10-24"]

def pq_path(name):
    return DATA_DIR / f"{name}.parquet"
import gc
import pyarrow.dataset as ds


In [2]:
FILES = [
    "clean_stop_visits", "expected_per_stop", "per_stop_evidence", "stop_event_status", "stop_gps_status", "time_lost_null", "trip_vehicle"
]

rows = []
for name in FILES:
    pf = pq.ParquetFile(pq_path(name))
    rows.append({
        "file": name,
        "rows": pf.metadata.num_rows,
        "cols": pf.metadata.num_columns,
        "row_groups": pf.metadata.num_row_groups,
        "columns": ", ".join(pf.schema_arrow.names),
    })
overview = pd.DataFrame(rows)
display(overview[["file", "rows", "cols", "row_groups"]])

# Full column lists, if you want to read them
for r in rows:
    print(f"{r['file']:<20} {r['columns']}")

,file,rows,cols,row_groups
0,clean_stop_visits,87279,30,1
1,expected_per_stop,13780,6,1
2,per_stop_evidence,2173,7,1
3,stop_event_status,87092,20,1
4,stop_gps_status,86241,29,1
5,time_lost_null,47480,1,1
6,trip_vehicle,2600,6,1


clean_stop_visits    service_date, journey_ref, line_ref, route_short_name, route_type_name, direction_ref, stop_order, stop_ref, stop_name, stop_lat, stop_lon, aimed_arrival, actual_arrival, aimed_departure, actual_departure, visit_time, aimed_time, delay_event, delay_s, delay_arrival_s, delay_departure_s, delay_outlier, on_time, is_actual, n_snapshots, local_date, local_hour, weekday, hour_utc, visit_time_local
expected_per_stop    service_date, stop_id, expected_visits, stop_name, stop_lat, stop_lon
per_stop_evidence    stop_id, calls, stopped, passed, stop_name, conclusive, passed_share
stop_event_status    trip_id, stop_id, arrival_time, departure_time, stop_sequence, pickup_type, drop_off_type, occ, sched_no_stop, route_id, service_id, route_short_name, sched_arrival, sched_departure, service_date, sched_time, visit_status, visit_time, n_snapshots, final_status
stop_gps_status      service_date, trip_id, route_id, route_short_name, stop_id, stop_name, stop_sequence, occ, sched_no